# Introdução aos Grafos Computacionais com PyTorch

> Parte da série [ML Notebooks](../README.md) — por **Nandobez**.



Neste notebook fornecemos uma breve introdução e visão geral de grafos computacionais usando PyTorch.

Existem muitos materiais online cobrindo a teoria, mas acho bem mais fácil aprender o conceito mexendo em código. Aqui tento fazer essa ponte, útil para iniciantes.

Inspirado pelo artigo do Olah ["Calculus on Computational Graphs: Backpropagation"](https://colah.github.io/posts/2015-08-Backprop/), juntei alguns trechos de código para você começar com grafos computacionais em PyTorch.


### Por que Grafos Computacionais?

Quando falamos de redes neurais em qualquer contexto, a [retropropagação](https://en.wikipedia.org/wiki/Backpropagation) é um tópico importante por ser o algoritmo usado no treinamento de redes profundas.

A retropropagação calcula as derivadas necessárias para continuar otimizando os parâmetros do modelo e fazendo-o aprender a tarefa em questão.

Vários frameworks modernos (incluindo PyTorch) fazem retropropagação automaticamente via [**diferenciação automática**](https://pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html).

Para entender melhor como isso é feito, vale falar sobre **grafos computacionais**, que definem o fluxo de operações pela rede. Vamos usar `torch.autograd` para demonstrar em código.


### Primeiros Passos

Inspirados pelo artigo do Olah, olhemos a expressão $e = (a + b) \cdot (b + 1)$. Quebrando em operações:

$$
\begin{aligned} c &= a + b \\ d &= b + 1 \\ e &= c \cdot d \end{aligned}
$$

Isto não é uma rede neural — é só uma cadeia simples de operações que pode ser representada como grafo computacional.

Vamos visualizar essas operações. Grafos computacionais têm **nós** (que representam uma entrada — tensor, matriz, vetor ou escalar — ou uma **operação** que pode ser entrada de outro nó), conectados por **arestas** que representam argumentos. Os grafos são *direcionados e acíclicos*. O grafo do nosso exemplo:

![](https://colah.github.io/posts/2015-08-Backprop/img/tree-def.png)

Podemos avaliar a expressão definindo, por exemplo, $a = 2$ e $b = 1$, e propagando os valores pelo grafo.

Em vez de fazer isso à mão, podemos usar o motor de diferenciação automática do PyTorch.

Vamos primeiro importar o PyTorch:


In [ ]:
import torch

Definimos as entradas:


In [ ]:
a = torch.tensor([2.], requires_grad=True)
b = torch.tensor([1.], requires_grad=True)

Note que usamos `requires_grad=True` para que o autograd rastreie toda operação sobre esses tensores.

Estas são as operações em código:


In [ ]:
c = a + b
d = b + 1
e = c * d

# grads populated for non-leaf nodes
c.retain_grad()
d.retain_grad()
e.retain_grad()

Note que usamos `.retain_grad()` para que os gradientes sejam guardados também em nós intermediários (não-folhas), já que queremos inspecioná-los.

Agora que temos o grafo, podemos checar o resultado da expressão:


In [ ]:
print(e)

A saída é um tensor com valor `6.`, que confere com o resultado esperado:

![](https://colah.github.io/posts/2015-08-Backprop/img/tree-eval.png)


### Derivadas em Grafos Computacionais

Usando o conceito de grafo computacional, queremos agora avaliar **derivadas parciais** ao longo das arestas. Isso nos dá os gradientes — que são o que se usa para treinar uma rede neural — e tudo isso pode ser feito pelo motor de diferenciação automática.

A intuição: queremos saber, por exemplo, se $a$ afeta diretamente $c$, e quanto. Em outras palavras, se mudarmos $a$ um pouquinho, quanto $c$ muda? Isso é a derivada parcial de $c$ em relação a $a$.

Dá pra fazer à mão, mas no PyTorch basta chamar `.backward()` em $e$ e deixar o motor descobrir os valores. `.backward()` sinaliza ao autograd para calcular os gradientes e armazená-los em `.grad`.


In [ ]:
e.backward()

Agora, queremos a derivada de $e$ em relação a $a$, ou seja, $\dfrac{\partial e}{\partial a}$.

Acessamos via `.grad`:


In [ ]:
print(a.grad)

A intuição por trás disso (parafraseando Olah):

> Vejamos como $e$ é afetado por $a$. Se mudamos $a$ a uma taxa $1$, $c$ também muda a uma taxa $1$. Por sua vez, $c$ mudando a taxa $1$ faz $e$ mudar a uma taxa $2$. Logo, $e$ varia a $1 \cdot 2$ em relação a $a$.

Em outras palavras, à mão:

$$
\frac{\partial e}{\partial a} = \frac{\partial e}{\partial c}\,\frac{\partial c}{\partial a} = 2 \cdot 1
$$

Como $a$ não está diretamente ligado a $e$, somamos a contribuição por todos os caminhos no grafo, multiplicando as derivadas em cada aresta.

![](https://colah.github.io/posts/2015-08-Backprop/img/tree-eval-derivs.png)


Para confirmar, vejamos outro exemplo: $\dfrac{\partial e}{\partial b}$.

Obtido via `b.grad`:


In [ ]:
print(b.grad)

À mão, é basicamente:

$$
\frac{\partial e}{\partial b} = 1 \cdot 2 + 1 \cdot 3
$$

Isso indica como $b$ afeta $e$ por dois caminhos: via $c$ e via $d$. Estamos somando contribuições.

Todos os gradientes coletados, incluindo nós intermediários:


In [ ]:
print(a.grad, b.grad, c.grad, d.grad, e.grad)

Use o grafo computacional acima para verificar. Esse é o poder dos grafos computacionais e dos motores de diferenciação automática. Também é um conceito útil ao projetar arquiteturas de rede neural.

### Próximos Passos

Recomendo fortemente a leitura do [artigo do Olah](https://colah.github.io/posts/2015-08-Backprop/). Outros recursos úteis:

- [Hacker's guide to Neural Networks](http://karpathy.github.io/neuralnets/)
- [Backpropagation calculus](https://www.youtube.com/watch?v=tIeHLnjs5U8) (3Blue1Brown)


## References

- Series repo: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Author: [Nandobez](https://github.com/Nandobez)
